# Stage 2 Notebook 59 - Exp2DDD DN-DETR + 10ep head_warmup + slow backbone

**Save the trapped potential.** NB56 (DN-DETR + 30 ep) hit val_lane_best_f1=0.588 and pos-neg gap=0.328 at EPOCH 1 of head_warmup (backbone frozen). Both metrics DEGRADED as training continued past the phase boundary at epoch 4. The peak best_f1 ended up at 0.468 by epoch 30 -- a 20% regression from the peak.

Diagnosis: head_warmup (3 epochs) was too short. The head's cls converged to a strong discriminative state on the frozen-random backbone, but when backbone unfroze at epoch 4 the backbone updates destabilized the head's learned cls patterns. With more head_warmup AND slower backbone unfreeze, the head should retain its peak.

Diffs vs NB56 (exp51):
- head_warmup until_epoch: 3 -> 10 (head fully converges before backbone is allowed to move)
- backbone_lr_mult: 0.1 -> 0.02 (5x slower than NB56, 10x slower than baseline)
- end_epoch: 30 -> 20 (we now expect peak earlier with the longer warmup)
- warmup_epochs (LR scheduler): 3 (unchanged)

Reference for the recipe: DETR's original training (Carion 2020) used `lr_backbone=1e-6` while main LR was 1e-4 -- 100x slower backbone. We're at 50x slower with lr0=2e-4 + bb_mult=0.02 -> backbone LR = 4e-6.

### Run mode
1. Smoke first.
2. 20 epochs limit=3000.

In [1]:
import os, sys, subprocess, textwrap
from google.colab import drive
os.environ['PYTHONUNBUFFERED'] = '1'
drive.mount('/content/drive')

REPO_ROOT = '/content/drive/MyDrive/EcoCAR/yolop_vehicle_lane'
if not os.path.isdir(REPO_ROOT):
    raise FileNotFoundError(f'Missing project root: {REPO_ROOT}')
os.chdir(REPO_ROOT)
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'pyyaml', 'scipy', 'opencv-python-headless', 'tqdm', 'matplotlib'])
print('repo:', REPO_ROOT)

from stage2.scripts.notebook_utils import run_streaming
LOG_DIR = '/content/drive/MyDrive/EcoCAR/training_runs/notebook_logs'
os.makedirs(LOG_DIR, exist_ok=True)

Mounted at /content/drive
repo: /content/drive/MyDrive/EcoCAR/yolop_vehicle_lane


In [2]:
from pathlib import Path
import os, sys

CONFIG = 'stage2/configs/exp54_rmt_gca_query64_dn_vfl_long_warmup_joint.yaml'
LOG_FILE = os.path.join(LOG_DIR, f'{Path(CONFIG).stem}_smoke.log')
run_streaming([sys.executable, '-u', 'stage2/scripts/smoke_test_joint_models.py', CONFIG], log_path=LOG_FILE)

[run_streaming] command: /usr/bin/python3 -u stage2/scripts/smoke_test_joint_models.py stage2/configs/exp54_rmt_gca_query64_dn_vfl_long_warmup_joint.yaml
[run_streaming] log file: /content/drive/MyDrive/EcoCAR/training_runs/notebook_logs/exp54_rmt_gca_query64_dn_vfl_long_warmup_joint_smoke.log
OK exp54_rmt_gca_query64_dn_vfl_long_warmup_joint.yaml
  lane_shape=(1, 64, 72, 2) det_shape=(1, 4, 4)
  lane_loss=3.3297 det_loss=3.3414 grad_cos=-0.1983 lambda_lane=0.2470
  gate_stats={'gate/det_mean': 0.5010926127433777, 'gate/lane_mean': 0.49876442551612854, 'gate/det_sat_low': 0.0, 'gate/det_sat_high': 0.0, 'gate/lane_sat_low': 0.0, 'gate/lane_sat_high': 0.0}
[run_streaming] return_code=0


0

In [3]:
from pathlib import Path
import os, sys

CONFIG = 'stage2/configs/exp54_rmt_gca_query64_dn_vfl_long_warmup_joint.yaml'
CURVE_TAR = '/content/drive/MyDrive/EcoCAR/datasets/bdd100k_clrkd_curve.tar'
CURVE_ROOT = '/content/bdd100k_clrkd_curve'

DEBUG_MODE = False

if DEBUG_MODE:
    RUN_TAG = 'debug'
    EPOCHS = 2
    BATCH_SIZE = 4
    LIMIT_TRAIN = 512
    LIMIT_VAL = 256
    PRINT_EVERY = 5
else:
    RUN_TAG = 'short20'
    EPOCHS = 20
    BATCH_SIZE = 8
    LIMIT_TRAIN = 3000
    LIMIT_VAL = 1000
    PRINT_EVERY = 50

run_stem = Path(CONFIG).stem + '_' + RUN_TAG
WORK_DIR = f'/content/{run_stem}'
OUTPUT_TAR = f'/content/drive/MyDrive/EcoCAR/training_runs/{run_stem}.tar'
LOG_FILE = os.path.join(LOG_DIR, f'{run_stem}_train.log')

cmd = [
    sys.executable, '-u', 'stage2/scripts/train_joint_model_experiment.py',
    '--config', CONFIG,
    '--curve-tar', CURVE_TAR,
    '--curve-root', CURVE_ROOT,
    '--work-dir', WORK_DIR,
    '--output-tar', OUTPUT_TAR,
    '--epochs', str(EPOCHS),
    '--batch-size', str(BATCH_SIZE),
    '--limit-val', str(LIMIT_VAL),
    '--force-extract',
    '--print-every', str(PRINT_EVERY),
]
if LIMIT_TRAIN is not None:
    cmd.extend(['--limit-train', str(LIMIT_TRAIN)])

print('DEBUG_MODE:', DEBUG_MODE, flush=True)
print('LIMIT_TRAIN:', LIMIT_TRAIN, flush=True)
print('About to run:', ' '.join(cmd), flush=True)
print('Output tar:', OUTPUT_TAR, flush=True)
print('Visible log file:', LOG_FILE, flush=True)
run_streaming(cmd, log_path=LOG_FILE)

DEBUG_MODE: False
LIMIT_TRAIN: 3000
About to run: /usr/bin/python3 -u stage2/scripts/train_joint_model_experiment.py --config stage2/configs/exp54_rmt_gca_query64_dn_vfl_long_warmup_joint.yaml --curve-tar /content/drive/MyDrive/EcoCAR/datasets/bdd100k_clrkd_curve.tar --curve-root /content/bdd100k_clrkd_curve --work-dir /content/exp54_rmt_gca_query64_dn_vfl_long_warmup_joint_short20 --output-tar /content/drive/MyDrive/EcoCAR/training_runs/exp54_rmt_gca_query64_dn_vfl_long_warmup_joint_short20.tar --epochs 20 --batch-size 8 --limit-val 1000 --force-extract --print-every 50 --limit-train 3000
Output tar: /content/drive/MyDrive/EcoCAR/training_runs/exp54_rmt_gca_query64_dn_vfl_long_warmup_joint_short20.tar
Visible log file: /content/drive/MyDrive/EcoCAR/training_runs/notebook_logs/exp54_rmt_gca_query64_dn_vfl_long_warmup_joint_short20_train.log
[run_streaming] command: /usr/bin/python3 -u stage2/scripts/train_joint_model_experiment.py --config stage2/configs/exp54_rmt_gca_query64_dn_vfl_lo

0

## What to watch in Exp2DDD

Reference NB56 epoch 1 (peak): val_lane_best_f1=0.588, val_lane_f1=0.451, gap=0.328, matched_iou=0.147.
Reference NB56 epoch 30: val_lane_best_f1=0.468, gap=0.109, matched_iou=0.139.

Pass criteria at epoch 20:
- **val/lane_best_f1 stays >= 0.50 from epoch 10 onwards** -- the slow backbone preserves the head's peak cls discrimination.
- val/lane_f1 >= 0.40.
- pos_score - neg_score >= 0.10 at epoch 20 (NB56 was 0.10).
- matched_iou >= 0.20 (DN queries get more epochs to refine geometry).
- val_det <= 2.5 (slow backbone should prevent the NB56 epoch-7 det collapse).

If val_lane_best_f1 stays high: this is the DEPLOYABLE query-head model. Geometry weak but cls strong; combine with anchor-head curves at inference in Exp2FFF.

If val_lane_best_f1 still degrades: peak-vs-decay isn't about the phase transition; query embeddings inherently drift. Pivot to EMA model averaging.